In [0]:
%pip install xgboost

In [0]:
import pandas as pd
import numpy as np
import pickle
import boto3
import json
from io import BytesIO
from datetime import datetime, timezone

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error
from xgboost import XGBRegressor

# =============================================================
# 1. CARGAR DATOS — Silver deduped (preferido) ó Gold legacy
# =============================================================
# Credenciales: Databricks secrets (producción) → archivo local (desarrollo)
try:
    config = {
        "aws_access_key": dbutils.secrets.get(scope="aws", key="access_key"),
        "aws_secret_key": dbutils.secrets.get(scope="aws", key="secret_key"),
        "bucket_name": "bronce-scrap-date",
    }
except Exception:
    with open("aws_secrets.json", "r") as f:
        config = json.load(f)

BUCKET = config["bucket_name"]
S3_OPTS = {
    "fs.s3a.access.key": config["aws_access_key"],
    "fs.s3a.secret.key": config["aws_secret_key"],
    "fs.s3a.endpoint": "s3.amazonaws.com",
}

# Intentar silver/master_deduped (Delta) — tiene city_token, tipo normalizado, dedup
# Fallback a gold/app_inmuebles (Parquet) si no existe
try:
    ruta_deduped = f"s3a://{BUCKET}/silver/master_deduped/"
    reader = spark.read.format("delta")
    for k, v in S3_OPTS.items():
        reader = reader.option(k, v)
    df_spark = reader.load(ruta_deduped)
    print(f"📊 Leyendo Silver Deduped (Delta): {ruta_deduped}")
except Exception:
    ruta_gold = f"s3a://{BUCKET}/gold/app_inmuebles/"
    reader = spark.read.format("parquet")
    for k, v in S3_OPTS.items():
        reader = reader.option(k, v)
    df_spark = reader.load(ruta_gold)
    print(f"📊 Fallback → Gold legacy (Parquet): {ruta_gold}")

print(f"   Columnas: {df_spark.columns}")

# =============================================================
# 2. PREPARAR FEATURES
# =============================================================
cols_target = ["precio_num"]
cols_numeric = ["area_m2", "habitaciones", "banos", "garajes"]
# city_token: ~35 ciudades vs 8,430 zonas en ubicacion_raw
# tipo_inmueble: ya normalizado a ~5 categorías por dedup module
cols_categorical = ["tipo_inmueble", "estado_inmueble", "fuente", "city_token"]
# ubicacion_norm: sin acentos, lowercase, limpio → TF-IDF más consistente
cols_text = ["ubicacion_norm", "titulo"]

# Fallback: si no hay ubicacion_norm/city_token (Gold legacy), usar raw
available = set(df_spark.columns)
if "ubicacion_norm" not in available and "ubicacion_raw" in available:
    cols_text = ["ubicacion_raw", "titulo"]
if "city_token" not in available:
    cols_categorical = [c for c in cols_categorical if c != "city_token"]

cols_numeric = [c for c in cols_numeric if c in available]
cols_categorical = [c for c in cols_categorical if c in available]
cols_text = [c for c in cols_text if c in available]

all_cols = cols_target + cols_numeric + cols_categorical + cols_text
print(f"📋 Features: numeric={cols_numeric}, cat={cols_categorical}, text={cols_text}")

df = df_spark.select(*all_cols).toPandas()
print(f"   Total registros: {len(df)}")

# Construir feature de texto
text_parts = []
for c in cols_text:
    text_parts.append(df[c].fillna(""))
df["texto_completo"] = text_parts[0]
for part in text_parts[1:]:
    df["texto_completo"] = df["texto_completo"] + " " + part

# Limpiar categóricos
for c in cols_categorical:
    df[c] = df[c].fillna("desconocido")

# Colapsar city_token raros → "otra_ciudad" (evitar 455 categorías OneHot)
if "city_token" in df.columns:
    conteos = df["city_token"].value_counts()
    ciudades_validas = conteos[conteos >= 100].index
    df["city_token"] = df["city_token"].where(df["city_token"].isin(ciudades_validas), other="otra_ciudad")
    print(f"   city_token: {df['city_token'].nunique()} categorías después de colapso (umbral=100)")
    print(df["city_token"].value_counts().to_string())

# Limpiar numéricos
for c in cols_numeric:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# =============================================================
# 3. FILTRAR OUTLIERS — IQR por city_token (si disponible)
# =============================================================
# Pre-filtro: área razonable + precio positivo
df = df[(df["area_m2"] > 15) & (df["area_m2"] < 1000) & (df["precio_num"] > 0)].copy()

if "city_token" in df.columns:
    # IQR per city: no mezclar Soacha con El Poblado
    def _iqr_filter(group):
        q1 = group["precio_num"].quantile(0.10)
        q3 = group["precio_num"].quantile(0.90)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        return group[(group["precio_num"] >= lower) & (group["precio_num"] <= upper)]

    # Solo filtrar por IQR en ciudades con suficientes datos
    big_cities = df.groupby("city_token").filter(lambda g: len(g) >= 30)
    small_cities = df[~df.index.isin(big_cities.index)]
    # IQR en ciudades grandes, percentil global en ciudades chicas
    df_big = big_cities.groupby("city_token", group_keys=False).apply(_iqr_filter, include_groups=False)
    p05, p95 = small_cities["precio_num"].quantile(0.05), small_cities["precio_num"].quantile(0.95)
    df_small = small_cities[(small_cities["precio_num"] > p05) & (small_cities["precio_num"] < p95)]
    df_clean = pd.concat([df_big, df_small], ignore_index=True)
    print(f"🧹 Outlier filter (IQR per city): {len(df_clean)} registros (de {len(df)})")
else:
    # Fallback global
    df_clean = df[
        (df["precio_num"] > df["precio_num"].quantile(0.05)) &
        (df["precio_num"] < df["precio_num"].quantile(0.95))
    ].copy()
    print(f"🧹 Outlier filter (global p5-p95): {len(df_clean)} registros (de {len(df)})")

# =============================================================
# 3b. DIAGNÓSTICO — Distribución de precios (post-filtro)
# =============================================================
print("\n" + "=" * 60)
print("  DIAGNÓSTICO DE DISTRIBUCIÓN DE PRECIOS")
print("=" * 60)

print("\n=== Distribución de precio_num ===")
print(df_clean["precio_num"].describe(percentiles=[.05,.10,.25,.50,.75,.90,.95,.99]).to_string())
print(f"\n  Mediana:  ${df_clean['precio_num'].median()/1e6:.0f}M")
print(f"  Media:    ${df_clean['precio_num'].mean()/1e6:.0f}M")
print(f"  Skewness: {df_clean['precio_num'].skew():.2f}")

if "city_token" in df_clean.columns:
    print("\n=== Distribución por ciudad (top 15) ===")
    city_stats = (df_clean.groupby("city_token")["precio_num"]
        .agg(["median", "mean", "count", "std"])
        .sort_values("count", ascending=False))
    city_display = city_stats.copy()
    for col in ["median", "mean", "std"]:
        city_display[col] = (city_display[col] / 1e6).round(0)
    print(city_display.head(15).to_string())

print("\n=== Concentración por rango de precio ===")
rangos = [0, 200e6, 400e6, 600e6, 800e6, 1000e6, 1500e6, 2000e6, float("inf")]
etiquetas = ["<200M", "200-400M", "400-600M", "600-800M", "800M-1B", "1B-1.5B", "1.5B-2B", ">2B"]
df_clean["_rango_precio"] = pd.cut(df_clean["precio_num"], bins=rangos, labels=etiquetas)
rango_counts = df_clean["_rango_precio"].value_counts().sort_index()
rango_pct = (rango_counts / len(df_clean) * 100).round(1)
print(pd.DataFrame({"n": rango_counts, "pct": rango_pct}).to_string())
df_clean = df_clean.drop(columns=["_rango_precio"])

print(f"\n  > $1,000M: {(df_clean['precio_num'] > 1e9).sum():,} ({(df_clean['precio_num'] > 1e9).mean()*100:.1f}%)")
print(f"  > $2,000M: {(df_clean['precio_num'] > 2e9).sum():,} ({(df_clean['precio_num'] > 2e9).mean()*100:.1f}%)")
print("=" * 60)

# =============================================================
# 4. FEATURES DE CONTEXTO DE MERCADO + PIPELINE ML
# =============================================================
# precio_m2_mediano_ciudad: ¿cuánto cuesta el m² típico en esta ciudad?
# precio_mediano_ciudad:    ¿cuánto cuesta un inmueble típico en esta ciudad?
# Estas features resuelven el problema de que area_m2 sola no distingue
# un apto de 80m² en Soacha ($200M) de uno en Chapinero ($600M).
#
# IMPORTANTE: se calculan SOLO en el training set para evitar data leakage,
# y luego se mapean al test set (con fallback a la mediana global).

# Seleccionar features numéricas con suficientes datos (>30% no-nulos)
num_features_final = []
for c in cols_numeric:
    pct_valid = df_clean[c].notna().mean()
    if pct_valid > 0.3:
        num_features_final.append(c)
        print(f"  ✅ {c}: {pct_valid:.0%} válidos")
    else:
        print(f"  ⏩ {c}: {pct_valid:.0%} válidos (descartado, <30%)")

# Seleccionar features categóricas con variabilidad
cat_features_final = []
for c in cols_categorical:
    n_unique = df_clean[c].nunique()
    if n_unique > 1:
        cat_features_final.append(c)
        print(f"  ✅ {c}: {n_unique} categorías")

# ── Train/test split ANTES de calcular features de mercado ──
feature_cols_base = num_features_final + cat_features_final + ["texto_completo"]
X = df_clean[feature_cols_base + (["city_token"] if "city_token" not in feature_cols_base else [])]
y = df_clean["precio_num"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ── Calcular features de mercado SOLO desde el training set ──
market_features_added = []
if "city_token" in X_train.columns:
    train_with_price = X_train.copy()
    train_with_price["_precio"] = y_train.values

    # Precio mediano por m² en cada ciudad (solo train)
    pm2_city = (
        train_with_price
        .dropna(subset=["area_m2"])
        .assign(_pm2=lambda d: d["_precio"] / d["area_m2"])
        .groupby("city_token")["_pm2"]
        .median()
    )
    global_pm2 = (y_train / X_train["area_m2"].dropna()).median()

    # Precio mediano absoluto por ciudad (solo train)
    pmed_city = train_with_price.groupby("city_token")["_precio"].median()
    global_pmed = y_train.median()

    # Mapear a train y test (test usa los valores del train; unseen cities → global)
    X_train = X_train.copy()
    X_test = X_test.copy()
    X_train["precio_m2_mediano_ciudad"] = X_train["city_token"].map(pm2_city).fillna(global_pm2)
    X_test["precio_m2_mediano_ciudad"] = X_test["city_token"].map(pm2_city).fillna(global_pm2)
    X_train["precio_mediano_ciudad"] = X_train["city_token"].map(pmed_city).fillna(global_pmed)
    X_test["precio_mediano_ciudad"] = X_test["city_token"].map(pmed_city).fillna(global_pmed)

    market_features_added = ["precio_m2_mediano_ciudad", "precio_mediano_ciudad"]
    num_features_final.extend(market_features_added)
    print(f"\n  🏙️  Features de mercado añadidas: {market_features_added}")
    print(f"      precio_m2_mediano_ciudad — rango: ${pm2_city.min()/1e6:.1f}M – ${pm2_city.max()/1e6:.1f}M por m²")
    print(f"      precio_mediano_ciudad   — rango: ${pmed_city.min()/1e6:.0f}M – ${pmed_city.max()/1e6:.0f}M")
    print(f"      (calculadas sin data leakage: solo desde training set)")

# Features finales
feature_cols = num_features_final + cat_features_final + ["texto_completo"]
# Asegurar que X_train/X_test solo tengan las columnas del pipeline
X_train = X_train[[c for c in feature_cols if c in X_train.columns]]
X_test = X_test[[c for c in feature_cols if c in X_test.columns]]

# Construir transformadores
transformers = [
    ("txt", TfidfVectorizer(
        max_features=300,
        stop_words=["en", "de", "la", "el", "y", "con", "se", "del", "por", "los", "las", "un", "una"]
    ), "texto_completo"),
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
    ]), num_features_final),
]

if cat_features_final:
    transformers.append(
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="desconocido")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), cat_features_final)
    )

preprocessor = ColumnTransformer(transformers=transformers)

# XGBoost con log-transform en target (precios tienen distribución log-normal)
xgb = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
)

model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", TransformedTargetRegressor(
        regressor=xgb, func=np.log1p, inverse_func=np.expm1
    )),
])

# =============================================================
# 5. ENTRENAR + EVALUAR
# =============================================================
print("\n🚀 Entrenando XGBoost con features enriquecidas...")
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
r2 = model.score(X_test, y_test)
mae = mean_absolute_error(y_test, y_pred)
mape = mean_absolute_percentage_error(y_test, y_pred) * 100

print(f"\n📊 RESULTADOS:")
print(f"   🌟 R² Score:  {r2:.4f}")
print(f"   💰 MAE:       ${mae:,.0f} COP")
print(f"   📏 MAPE:      {mape:.1f}%")

# Cross-validation (5-fold) — Note: CV R² may be slightly optimistic because
# market features (precio_m2/mediano_ciudad) were computed on the full train set.
# The hold-out test metrics above remain leak-free, which is what matters.
print("\n⏳ Cross-validation (5-fold)...")
cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring="r2")
print(f"   CV R²: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

# =============================================================
# 5b. DIAGNÓSTICO — MAPE por rango de precio
# =============================================================
print("\n" + "=" * 60)
print("  MAPE POR RANGO DE PRECIO (diagnóstico)")
print("=" * 60)
df_eval = pd.DataFrame({"real": y_test.values, "pred": y_pred})
df_eval["rango"] = pd.cut(df_eval["real"],
    bins=[0, 300e6, 500e6, 800e6, 1500e6, float("inf")],
    labels=["<300M", "300-500M", "500-800M", "800M-1.5B", ">1.5B"])
mape_by_range = df_eval.groupby("rango", observed=True).apply(
    lambda x: pd.Series({
        "n": len(x),
        "mape_%": (abs(x.pred - x.real) / x.real * 100).mean().round(1),
        "mae_M": (abs(x.pred - x.real).mean() / 1e6).round(0),
        "precio_medio_M": (x.real.mean() / 1e6).round(0),
    }), include_groups=False
)
print(mape_by_range.to_string())
print(f"\n{'─'*60}")
n_luxury = (df_eval["real"] > 800e6).sum()
mape_sin_lujo = (abs(df_eval.loc[df_eval["real"] <= 800e6, "pred"] -
                       df_eval.loc[df_eval["real"] <= 800e6, "real"]) /
                  df_eval.loc[df_eval["real"] <= 800e6, "real"]).mean() * 100
print(f"  MAPE sin lujo (≤$800M): {mape_sin_lujo:.1f}%")
print(f"  Registros lujo (>$800M): {n_luxury} ({n_luxury/len(df_eval)*100:.1f}% del test)")
print("=" * 60)

# =============================================================
# 5c. DIAGNÓSTICO — Importancia de features
# =============================================================
print("\n" + "=" * 60)
print("  IMPORTANCIA DE FEATURES")
print("=" * 60)

xgb_model = model.named_steps["regressor"].regressor_
try:
    feature_names = model.named_steps["preprocessor"].get_feature_names_out()
except Exception:
    feature_names = [f"f{i}" for i in range(xgb_model.n_features_in_)]

importances = pd.Series(
    xgb_model.feature_importances_, index=feature_names
).sort_values(ascending=False)

print("\n=== Top 30 features por importancia ===")
print(importances.head(30).to_string())

print("\n=== Importancia por grupo ===")
groups = {
    "area_m2": importances.filter(like="num__area_m2").sum(),
    "habitaciones": importances.filter(like="num__habitaciones").sum(),
    "banos": importances.filter(like="num__banos").sum(),
    "garajes": importances.filter(like="num__garajes").sum(),
    "city_token (OHE)": importances.filter(like="cat__city_token").sum(),
    "tipo_inmueble": importances.filter(like="cat__tipo_inmueble").sum(),
    "fuente": importances.filter(like="cat__fuente").sum(),
    "estado_inmueble": importances.filter(like="cat__estado_inmueble").sum(),
    "texto_tfidf": importances.filter(like="txt__").sum(),
    "precio_m2_med_ciudad": importances.filter(like="num__precio_m2_mediano").sum(),
    "precio_med_ciudad": importances.filter(like="num__precio_mediano").sum(),
}
for name, imp in sorted(groups.items(), key=lambda x: -x[1]):
    if imp > 0:
        print(f"  {name:25s}: {imp:.4f}  ({imp*100:.1f}%)")

print(f"\n=== Correlación features numéricas con log(precio) ===")
df_corr = df_clean.copy()
df_corr["log_precio"] = np.log1p(df_corr["precio_num"])
for col in num_features_final:
    if col in df_corr.columns:
        r = df_corr[col].corr(df_corr["log_precio"])
        print(f"  {col:30s}: r = {r:.3f}")
print("=" * 60)

# =============================================================
# 6. GUARDAR MODELO EN S3 (Champion / Challenger)
# =============================================================
if r2 > 0.0:
    s3 = boto3.client(
        "s3",
        aws_access_key_id=config["aws_access_key"],
        aws_secret_access_key=config["aws_secret_key"],
    )

    # 1. Nombre versionado — nunca sobreescribe
    ts = datetime.now(tz=timezone.utc).strftime("%Y%m%d_%H%M%S")
    file_key = f"models/modelo_xgboost_{ts}.pkl"

    buffer = BytesIO()
    pickle.dump(model, buffer)
    buffer.seek(0)
    s3.upload_fileobj(buffer, BUCKET, file_key)
    print(f"\n💾 Modelo guardado: s3://{BUCKET}/{file_key}")

    # 2. Leer manifest anterior para comparar con campeón
    manifest_key = "models/manifest.json"
    try:
        prev = json.loads(
            s3.get_object(Bucket=BUCKET, Key=manifest_key)["Body"].read()
        )
        prev_mape = prev.get("metrics", {}).get("mape", float("inf"))
    except s3.exceptions.NoSuchKey:
        prev_mape = float("inf")
        prev = {}

    # 3. Solo actualizar el manifest si este modelo es mejor (o es el primero)
    MAPE_THRESHOLD = 0.5  # pp mínimos de mejora para desplazar al campeón

    if (prev_mape - mape) >= MAPE_THRESHOLD or prev_mape == float("inf"):
        manifest = {
            "champion_model_key": file_key,
            "champion_model_uri": f"s3://{BUCKET}/{file_key}",
            "metrics": {
                "r2": r2,
                "mae": mae,
                "mape": mape,
                "cv_r2_mean": float(cv_scores.mean()),
                "cv_r2_std": float(cv_scores.std()),
                "train_size": len(X_train),
                "test_size": len(X_test),
                "feature_cols": feature_cols,
            },
            "deployed_at": datetime.now(tz=timezone.utc).isoformat(),
            "previous_champion": prev.get("champion_model_key"),
            "previous_mape": prev_mape if prev_mape != float("inf") else None,
        }
        s3.put_object(
            Bucket=BUCKET,
            Key=manifest_key,
            Body=json.dumps(manifest, indent=2).encode(),
            ContentType="application/json",
        )
        print(f"✅ Nuevo campeón desplegado — MAPE: {mape:.2f}% (antes: {prev_mape:.2f}%)")
        print(f"   Manifest: s3://{BUCKET}/{manifest_key}")
    else:
        print(f"⏭️  Modelo NO desplegado — MAPE {mape:.2f}% no mejora al campeón {prev_mape:.2f}%")
        print(f"   El pickle fue guardado en S3 pero el manifest NO fue actualizado.")
        print(f"   Campeón activo sigue siendo: {prev.get('champion_model_key')}")